In [1]:
import phonlp
from graph.src.triplet_extraction import init_vncorenlp

vncorenlp_client = init_vncorenlp(r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-1.2")
phoNLP_model = phonlp.load(save_dir=r"E:\Github\uit_chatbot\graph\phonlp")

Loading model from: E:\Github\uit_chatbot\graph\phonlp/phonlp.pt


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2daa7055-9912-448a-9482-c101a874ee6d)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base/c1e37c5c86f918761049cef6fa216b4779d0d01d/config.json
Retrying in 1s [Retry 1/5].


In [2]:
from graph.src.db import init_sqlite

process_conn, process_cursor = init_sqlite(r"E:\Github\uit_chatbot\graph\jupyter\uit_law.db")

In [3]:
import logging

def setup_logger(name="triplet_extraction", level=logging.INFO, log_to_file=False):
    """
    Sets up a logger suitable for Jupyter Notebook.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Remove existing handlers to avoid duplicate logs
    if logger.hasHandlers():
        logger.handlers.clear()

    # ✅ Jupyter-friendly StreamHandler (prints cleanly in notebook cells)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(console_handler)

    # ✅ Optional file logging
    if log_to_file:
        file_handler = logging.FileHandler("triplet_extraction.log", encoding="utf-8")
        file_handler.setFormatter(
            logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
        )
        logger.addHandler(file_handler)

    return logger

# Example: create global logger
logger = setup_logger(level=logging.INFO)

In [4]:


def parse_dataframe_to_tokens(df):
    """
    Convert DataFrame to a list of token dicts.
    """
    tokens = []
    for _, row in df.iterrows():
        token = {
            'id': int(row['id']),
            'word': str(row['word']),
            'pos': str(row['pos']),
            'head': int(row['head']),
            'deprel': str(row['deprel'])
        }
        tokens.append(token)
    return tokens


def split_sentence_np_vp(tokens):
    if not tokens:
        return [], []

    root_index = -1
    root_id = None

    # Step 1: find the main verb (root or first valid verb)
    for i, token in enumerate(tokens):
        if token['pos'] == 'V':
            if token['deprel'] == 'root' and token['head'] == 0:
                # Avoid picking verb at start (index 0)
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']
                break
            elif root_index == -1 and token['deprel'] != 'nmod':
                # Avoid first word if it's a verb
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']

    # Step 2: fallback – pick next verb if root not found
    if root_index == -1:
        for i, token in enumerate(tokens):
            if token['pos'] == 'V' and i > 0:  # skip first position
                root_index = i
                root_id = token['id']
                break

    # Step 3: final split
    if root_index != -1 and root_id is not None:
        np_tokens = tokens[:root_index]
        vp_tokens = tokens[root_index:]
        return np_tokens, vp_tokens

    return [], []

def collect_dependents(tokens, head_id):
    """Return set of token ids: head_id + all recursive dependents"""
    subtree = {head_id}
    result = []
    added = True
    while added:
        added = False
        for token in tokens:
            if token['head'] in subtree and token['id'] not in subtree:
                subtree.add(token['id'])
                result.append(token)
                added = True
    return result


def collect_direct_dependents(tokens, head_id):
    """Return list of token dicts that directly depend on head_id"""
    return [t for t in tokens if t['head'] == head_id]


def rebuild_phrase(tokens):
    # Sort tokens by their original position in the sentence and join them together
    tokens_sorted = sorted(tokens, key=lambda x: x['id'])
    phrase = " ".join(t['word'] for t in tokens_sorted)
    return phrase

def extract_main_subjects(np_tokens):
    if not np_tokens:
        return []

    sub_tokens = [t for t in np_tokens if t['deprel'] == 'sub']
    if not sub_tokens:
        sub_tokens = [t for t in np_tokens if t['deprel'] == 'root']
    if not sub_tokens:
        return []

    main_subjects = [sub_tokens[0]]
    main_subjects.extend(collect_direct_dependents(np_tokens, sub_tokens[0]['id']))
    for sub in main_subjects:
        if sub['pos'] == 'N' and sub['deprel'] == 'nmod':
            main_subjects.remove(sub)
    if len(main_subjects) == len(np_tokens):
        return [rebuild_phrase(np_tokens)]

    # Find Coordination Word (Cc, CH)
    coord_tokens = [t for t in np_tokens if t['pos'] in ['Cc', 'CH']]
    non_main_tokens = set()
    if len(coord_tokens) > 0:
        phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(np_tokens) if t['id'] == coord['id']), None)

            left_tokens = []
            main_subjects_id = [obj['id'] for obj in main_subjects]
            for i in range(coord_index - 1, -1, -1):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in main_subjects_id:
                    left_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            right_tokens = []
            for i in range(coord_index + 1, len(np_tokens)):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            if left_tokens:
                phrases.append(rebuild_phrase(left_tokens))
            if right_tokens:
                phrases.append(rebuild_phrase(right_tokens))

        # Remove duplicates while preserving order
        phrases = list(dict.fromkeys(phrases))

        for sub in main_subjects:
            if sub['id'] in non_main_tokens:
                main_subjects.remove(sub)
        main_subject_phrase = rebuild_phrase(main_subjects)

        # Properly combine main subject with each phrase
        combined_phrases = []
        for phrase in phrases:
            combined_phrases.append(main_subject_phrase + " " + phrase)
        return combined_phrases
    else:
        return [rebuild_phrase(np_tokens)]

def extract_verbs(vp_tokens):
    if not vp_tokens:
        return [], []

    # Find the root verb first
    root_verb = None
    for t in vp_tokens:
        if t['deprel'] == 'root' and t['head'] == 0 and t['pos'] == 'V':
            root_verb = t
            break

    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V' and t['deprel'] not in ['nmod', 'aux']:
                root_verb = t
                break

    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V':
                root_verb = t
                break

    # Fallback: first token
    if not root_verb:
        return [vp_tokens[0]['word']], [vp_tokens[0]]

    # Check for coordination markers (CH, Cc) that are direct dependents of root
    coord_markers = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH'] and t['head'] == root_verb['id']]

    if coord_markers:
        coordinated_verbs = [root_verb]
        for t in vp_tokens:
            if t['pos'] == 'V' and t['head'] == root_verb['id'] and t['deprel'] in ['vmod', 'conj']:
                coordinated_verbs.append(t)

        # Sort by ID to maintain order
        coordinated_verbs.sort(key=lambda x: x['id'])

        verb_phrases = []
        all_tokens = []

        for verb in coordinated_verbs:
            phrase_tokens = [verb]
            dependents = collect_direct_dependents(vp_tokens, verb['id'])

            # Keep only dependents that are not other coordinated verbs or coordination markers
            dependents = [d for d in dependents if d['id'] not in [v['id'] for v in coordinated_verbs]
                          and d['pos'] not in ['CH', 'Cc'] and d['deprel'] == 'vmod']

            phrase_tokens.extend(dependents)
            all_tokens.extend(phrase_tokens)

            verb_phrases.append({
                'text': rebuild_phrase(phrase_tokens),
                'tokens': phrase_tokens
            })

        return verb_phrases, all_tokens

    # Single verb: return it with its dependents
    verb_tokens = [root_verb]
    verb_tokens.extend(collect_direct_dependents(vp_tokens, root_verb['id']))
    sorted_verb_tokens = sorted(verb_tokens, key=lambda x: x['id'])

    # Filter out tokens after the first noun
    filtered_tokens = []
    for token in sorted_verb_tokens:
        if token['pos'].startswith('N'):
            break
        filtered_tokens.append(token)

    # If we filtered out everything, at least return the root verb
    if not filtered_tokens:
        filtered_tokens = [root_verb]

    return [{
        'text': rebuild_phrase(filtered_tokens),
        'tokens': filtered_tokens
    }], filtered_tokens

def extract_objects(vp_tokens, verb_token):
    # Remove verb tokens from vp_tokens
    for v in verb_token:
        for t in vp_tokens:
            if t['id'] == v['id']:
                vp_tokens.remove(t)
                break

    if not vp_tokens:
        return []

    # Find object tokens (dob, iob, pob)
    obj_tokens = [t for t in vp_tokens if t['deprel'] in ['dob', 'iob', 'pob']]
    if not obj_tokens:
        return []

    # Collect main object and its dependents
    main_objects = [obj_tokens[0]]
    main_objects.extend(collect_direct_dependents(vp_tokens, obj_tokens[0]['id']))

    if len(main_objects) == len(vp_tokens):
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]

    # Find coordination tokens
    coord_tokens = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH']]
    for obj in main_objects:
        if obj in coord_tokens:
            main_objects = []
            break

    if coord_tokens:
        combined_phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(vp_tokens) if t['id'] == coord['id']), None)

            # LEFT TOKENS
            left_tokens = []
            for i in range(coord_index - 1, -1, -1):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in [obj['id'] for obj in main_objects]:
                    left_tokens.append(token)
                else:
                    break
            left_tokens = left_tokens[::-1]

            # RIGHT TOKENS
            right_tokens = []
            for i in range(coord_index + 1, len(vp_tokens)):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                else:
                    break

            # Combine main object with left and right tokens
            for tokens_side in [left_tokens, right_tokens]:
                if tokens_side:
                    combined_phrases.append({
                        'text': rebuild_phrase(main_objects) + " " + rebuild_phrase(tokens_side),
                        'tokens': main_objects + tokens_side
                    })

        # Remove duplicates while preserving order
        seen = set()
        final_phrases = []
        for item in combined_phrases:
            if item['text'] not in seen:
                final_phrases.append(item)
                seen.add(item['text'])

        return final_phrases

    else:
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]


def process_sentence(df):
    tokens = parse_dataframe_to_tokens(df)
    np_tokens, vp_tokens = split_sentence_np_vp(tokens)

    # Debug log
    logger.debug("-----------------NP-----------------")
    logger.debug(np_tokens)
    logger.debug("-----------------VP-----------------")
    logger.debug(vp_tokens)

    # Step 3: extract subjects, verbs, objects
    subjects = extract_main_subjects(np_tokens)                 # list of phrases
    verbs, verbs_token = extract_verbs(vp_tokens)               # list of verbs
    objects = extract_objects(vp_tokens, verbs_token)           # list of object phrases

    logger.debug("-----------------subjects----------------")
    logger.debug(subjects)
    logger.debug("-----------------verbs----------------")
    for verb in verbs:
        logger.debug(verb['text'])
    logger.debug("-----------------objects----------------")
    for obj in objects:
        logger.debug(obj['text'])

    # Step 4: combine them into triplets
    verbs_position = {}
    for verb in verbs:
        verb_last_id = verb['tokens'][0]['id']
        verbs_position[verb['text']] = verb_last_id

    triplets = []

    # Sort verbs and objects by token positions
    verbs_sorted = sorted(verbs, key=lambda v: v['tokens'][0]['id'])
    objects_sorted = sorted(objects, key=lambda o: o['tokens'][0]['id'])

    # Iterate over all subjects
    for subj in subjects:
        for i, verb in enumerate(verbs_sorted):
            verb_last_id = verb['tokens'][-1]['id']

            # Determine the next verb's first ID (or infinity if this is the last verb)
            next_verb_first_id = verbs_sorted[i + 1]['tokens'][0]['id'] if i + 1 < len(verbs_sorted) else float('inf')

            # Objects that come after this verb but before the next verb
            obj_candidates = []
            for obj in objects_sorted:
                obj_id = obj['tokens'][-1]['id']
                if verb_last_id < obj_id < next_verb_first_id:
                    obj_candidates.append(obj)

            for obj in obj_candidates:
                triplets.append((subj, verb['text'], obj['text']))

    return triplets

def triplet_extraction(text, phoNLP_model, stopwords, max_depth=2, depth=0):
    """
    Recursively extract triplets from text, including nested subjects/objects.
    """
    if depth > max_depth or not text.strip():
        return []

    sentence = clean_text(text)
    segmented_text = vncorenlp_client.word_segment(sentence)

    # Stopword filtering
    parts = segmented_text[0].split(" ")
    filtered_parts = [part for part in parts if is_valid_term(part, stopwords)]
    filtered_text = " ".join(filtered_parts)

    # Annotate filtered text
    annotation = phoNLP_model.annotate(text=filtered_text)
    df = parsing_result(annotation)

    triplets = process_sentence(df)
    all_triplets = []

    for subj, verb, obj in triplets:
        # --- Refine subject ---
        subj_annotation = phoNLP_model.annotate(text=subj)
        df_subj = parsing_result(subj_annotation)
        refined_subj_triplets = process_sentence(df_subj)
        if refined_subj_triplets:
            # Replace subject with first refined subject
            subj_refined = refined_subj_triplets[0][0]
        else:
            subj_refined = subj

        # --- Refine object ---
        obj_annotation = phoNLP_model.annotate(text=obj)
        df_obj = parsing_result(obj_annotation)
        refined_obj_triplets = process_sentence(df_obj)
        if refined_obj_triplets:
            # Replace object with first refined object
            obj_refined = refined_obj_triplets[0][0]
        else:
            obj_refined = obj

        # Add the main triplet with refined elements
        all_triplets.append((subj_refined, verb, obj_refined))

        # Also add any new triplets extracted from subject
        all_triplets.extend(refined_subj_triplets)

        # Also add any new triplets extracted from object
        all_triplets.extend(refined_obj_triplets)

    return all_triplets

from graph.src.triplet_extraction import load_stopwords, is_valid_term

stopwords = load_stopwords(r"E:\Github\uit_chatbot\graph\stopwords.csv")

# text = "Chương trình đào tạo của mỗi ngành đào tạo do trường xây dựng phù hợp với các quy định hiện hành của Bộ GD&ĐT và ĐHQG-HCM, được bổ sung nội dung xây dựng kế hoạch và thực hiện các điều kiện đảm bảo chất lượng giáo dục"

text = "Chương trình song ngành cho phép sinh viên đủ điều kiện học cùng lúc hai chương trình đào tạo để khi tốt nghiệp được cấp hai văn bằng"
# Tạm thời stopwords rỗng
triplets = triplet_extraction(text, phoNLP_model, set(), max_depth=2)

logger.debug("Extracted Triplets:")
for triplet in triplets:
    logger.debug(triplet)

100%|██████████| 1/1 [00:00<00:00, 14.05it/s]


In [12]:
from graph.src.db import extract_random_rows

row = extract_random_rows(process_cursor, "laws_process", limit=1)[0]
_id = row["id"]
so_hieu = row["so_hieu"]
sentence = row["content"]
logger.debug(_id)
logger.debug(so_hieu)
logger.debug(sentence)

33538215fa393e13e2cf3e85247e2349304325222cd58434c2f77d044619d366
790/QĐ-ĐHCNTT
Hiệu trưởng ra quyết định bổ nhiệm cố vấn học tập


In [5]:


# row = extract_random_rows(process_cursor, "laws_process", limit=1)[0]
_id = row["id"]
so_hieu = row["so_hieu"]
sentence = row["content"]
logger.debug(_id)
logger.debug(so_hieu)
logger.debug(sentence)

from graph.src.triplet_extraction import load_stopwords, is_valid_term, parsing_result
from graph.src.triplet_extraction.utils import clean_text

stopwords = load_stopwords(r"E:\Github\uit_chatbot\graph\stopwords.csv")
text = sentence

sentence = clean_text(text)
logger.debug(f"Cleaned Sentence:\n {sentence}")

triplets = triplet_extraction(text, phoNLP_model, set(), max_depth=2)
logger.debug("Extracted Triplet:")
for t in triplets:
    logger.debug(t)

NameError: name 'row' is not defined